# 물리 정보 신경망 실습

**Physics-Informed Neural Network · PINN**

지배 방정식이나 경계 조건을 손실 함수에 넣어 물리 제약을 지키게 학습하는 신경망.

소재 분야에서 이해하기: 확산 방정식을 만족하도록 농도 분포를 학습한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [물리 정보 신경망 원논문](https://arxiv.org/abs/1711.10561)

## 1. 방정식을 손실에 넣기

정상 확산 방정식 u''(x) = -pi^2 sin(pi x), u(0)=u(1)=0 을 데이터 없이 풉니다.
참해는 sin(pi x) 입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from scipy.optimize import least_squares

hidden = 12

def network(params, x):
    a, b, w = params[:hidden], params[hidden:2 * hidden], params[2 * hidden:]
    return np.tanh(np.outer(x, a) + b) @ w

def second_derivative(params, x, h=1e-3):
    return (network(params, x + h) - 2 * network(params, x) + network(params, x - h)) / h ** 2

collocation = np.linspace(0.02, 0.98, 60)
source = -np.pi ** 2 * np.sin(np.pi * collocation)

def residuals(params):
    physics = second_derivative(params, collocation) - source
    boundary = np.array([network(params, np.array([0.0]))[0], network(params, np.array([1.0]))[0]])
    return np.concatenate([physics, 30 * boundary])   # 경계 조건에 큰 가중치

start = np.random.default_rng(0).normal(0, 0.7, 3 * hidden)
solution = least_squares(residuals, start, max_nfev=4000)
print('최종 잔차 노름 %.4f' % np.linalg.norm(solution.fun))

In [ ]:
grid = np.linspace(0, 1, 200)
predicted = network(solution.x, grid)
truth = np.sin(np.pi * grid)
plt.plot(grid, truth, 'k--', label='exact sin(pi x)')
plt.plot(grid, predicted, label='PINN')
plt.scatter(collocation, np.zeros_like(collocation), s=6, c='red', label='collocation points')
plt.legend(); plt.xlabel('x'); plt.ylabel('u'); plt.show()
print('최대 오차 %.4f' % np.max(np.abs(predicted - truth)))
print('데이터 없이 방정식과 경계 조건만으로 해를 얻었습니다.')

## 2. 경계 조건 가중치를 바꿔보기

In [ ]:
for weight in (0.1, 1.0, 30.0):
    def residuals_weighted(params, weight=weight):
        physics = second_derivative(params, collocation) - source
        boundary = np.array([network(params, np.array([0.0]))[0], network(params, np.array([1.0]))[0]])
        return np.concatenate([physics, weight * boundary])
    result = least_squares(residuals_weighted, start, max_nfev=3000)
    error = np.max(np.abs(network(result.x, grid) - truth))
    print('경계 가중치 %5.1f -> 최대 오차 %.4f' % (weight, error))
print('\n물리 항과 경계 항의 균형이 PINN 학습의 실질적인 난점입니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#pinn)을 여세요.